# COMP0005 - GROUP COURSEWORK
# Experimental Evaluation of Search Data Structures and Algorithms

The cell below defines **AbstractSearchInterface**, an interface to support basic insert/search operations; you will need to implement this three times, to realise your three search data structures of choice among: (1) *2-3 Tree*, (2) *AVL Tree*, (3) *LLRB BST*; (4) *B-Tree*; and (5) *Scapegoat Tree*. <br><br>**Do NOT modify the next cell** - use the dedicated cells further below for your implementation instead. <br>

In [177]:
# DO NOT MODIFY THIS CELL

from abc import ABC, abstractmethod  

class AbstractSearchInterface(ABC):
    '''
    Abstract class to support search/insert operations (plus underlying data structure)
    
    '''
        
    @abstractmethod
    def insertElement(self, element):     
        '''
        Insert an element in a search tree
            Parameters:
                    element: string to be inserted in the search tree (string)

            Returns:
                    "True" after successful insertion, "False" if element is already present (bool)
        '''
        
        pass 
    

    @abstractmethod
    def searchElement(self, element):
        '''
        Search for an element in a search tree
            Parameters:
                    element: string to be searched in the search tree (string)

            Returns:
                    "True" if element is found, "False" otherwise (bool)
        '''

        pass

Use the cell below to define any auxiliary data structure and python function you may need. Leave the implementation of the main API to the next code cells instead.

In [ ]:
# ADD AUXILIARY DATA STRUCTURE DEFINITIONS AND HELPER CODE HERE
class Node:
    def __init__(self,isLeaf=True):
        self._keys = []
        self._children = []
        self._isLeaf = isLeaf

    def addKey(self,key, pos = None):
        if pos == None:
            pos = len(self._keys)
        self._keys.insert(pos,key)
    
    def addChild(self,child_node,pos=None):
        if pos == None:
            pos = len(self._children)
        self._children.insert(pos,child_node)

    def getChildIndex(self,child):
        try:
            return self._children.index(child)
        except:
            return False
    
    def removeChild(self,child):
        self._children.remove(child)

    def getKeys(self):
        return self._keys
    
    def getChildren(self):
        return self._children
    
    def isLeaf(self):
        return self._isLeaf
    
    def setLeaf(self,leaf):
        self._leaf = leaf



In [ ]:
#Henry
class TwoThreeTree(AbstractSearchInterface):
    def __init__(self):
        self._root = None
        
    def insertElementRecursion(self, element): 
        if self._root == None: #Case 1: Empty tree
            self._root = Node()
            self._root.addKey(element)
            return True
        
        def traverseTree(element,currentNode,currentPath=None): #Traverse tree recursively to find where to place node
            if currentPath == None:
                currentPath = []
            currentNodeKeys = currentNode.getKeys()
            if element in currentNodeKeys:
                return False
            currentPath.append(currentNode)
            if currentNode.isLeaf():
                return currentPath
            
            children = currentNode.getChildren()

            if len(currentNodeKeys) == 0 or element < currentNodeKeys[0]:
                return traverseTree(element,children[0],currentPath)
            elif len(currentNodeKeys) == 1 or element < currentNodeKeys[1]:
                return traverseTree(element,children[1],currentPath)
            else:
                return traverseTree(element,children[2],currentPath)
        
        def addKeyToNode(node,element):
            keys = node.getKeys()
            if len(keys) == 0 or element < keys[0]:
                node.addKey(element,0)
            elif len(keys) == 1 or element < keys[1]:
                node.addKey(element,1)
            else:
                node.addKey(element,2)
            return node.getKeys()
        
        def addToTree(element,path):
            currentNode = path.pop()
            currentNodeKeys = addKeyToNode(currentNode,element)
            if len(currentNodeKeys) <= 2: #Adding to a 2-node
                return
            currentNodeChildren = currentNode.getChildren()
            currentLeaf = currentNode.isLeaf()
            isRoot = len(path) == 0

            if isRoot: #Adding to a 3-node and its a root node
                parentNode = Node(isLeaf=False)
                addKeyToNode(parentNode,currentNodeKeys[1])
                self._root = parentNode
                currentNodeIndex = 0
            else: #Adding to a 3-node
                parentNode = path[-1]
                currentNodeIndex = parentNode.getChildIndex(currentNode) #To add the new nodes in the correct position

            node1 = Node(currentLeaf) #Create new nodes 
            node1.addKey(currentNodeKeys[0])
            node2 = Node(currentLeaf)
            node2.addKey(currentNodeKeys[2])
            if currentLeaf == False: #Connect the new nodes to the rest of the tree if not leaf
                node1.addChild(currentNodeChildren[0])
                node1.addChild(currentNodeChildren[1])
                node2.addChild(currentNodeChildren[2])
                node2.addChild(currentNodeChildren[3])
            parentNode.removeChild(currentNode)
            parentNode.addChild(node1,currentNodeIndex) #Add nodes in correct spot, if left, positions 0,1, if right, positions 1,2
            parentNode.addChild(node2,currentNodeIndex+1)
            if isRoot == False:
                addToTree(currentNodeKeys[1],path) #Add middle key to previous node/new root node

        pathToLeaf = traverseTree(element,self._root)
        if pathToLeaf == False:
            return False
        addToTree(element,pathToLeaf)
        return True

    def searchElementRecrusion(self, element):     
        def search(element,currentNode):
            if (currentNode == None): #Check its not an empty tree
                return False
            currentNodeKeys = currentNode.getKeys()
            if element in currentNodeKeys: #Base case: Stops the recrusion, found item
                return True
            if currentNode.isLeaf(): #Base case: Stops the recrusion, end of tree
                return False

            currentNodeChildren = currentNode.getChildren()

            if element < currentNodeKeys[0]:
                return search(element,currentNodeChildren[0])
            elif len(currentNodeKeys) == 1 or element < currentNodeKeys[1]:
                return search(element,currentNodeChildren[1])
            elif element >= currentNodeKeys[1]:
                return search(element,currentNodeChildren[2])
    
        return search(element,self._root)
    
    def searchElement(self, element): #Python is not good for recursion so its done iteratively instead
        if self._root == None: #Check its not an empty tree
            return False
        currentNode = self._root
        while True:
            currentNodeKeys = currentNode.getKeys()
            found = element in currentNodeKeys
            if found == True: #Found element, stop looping
                return True
            currentNodeChildren = currentNode.getChildren()
            if len(currentNodeChildren) == 0: #Reached leaf, stop looping
                return False
            if element < currentNodeKeys[0]:
                currentNode = currentNodeChildren[0]
            elif len(currentNodeKeys) == 1 or element < currentNodeKeys[1]:
                currentNode = currentNodeChildren[1]
            elif element >= currentNodeKeys[1]:
                currentNode = currentNodeChildren[2]

    def insertElement(self, element):
        if self._root == None:
            self._root = Node()
            self._root.addKey(element)
            return True

        def getPathToLeaf(element): 
            currentNode = self._root
            pathToLeaf = [currentNode]
            while True:
                currentNodeKeys = currentNode.getKeys()
                found = element in currentNodeKeys
                if found == True: #Found element, stop looping
                    return False
                currentNodeChildren = currentNode.getChildren()
                if currentNode.isLeaf(): #Reached leaf, stop looping
                    return pathToLeaf
                if element < currentNodeKeys[0]:
                    currentNode = currentNodeChildren[0]
                elif len(currentNodeKeys) == 1 or element < currentNodeKeys[1]:
                    currentNode = currentNodeChildren[1]
                elif element >= currentNodeKeys[1]:
                    currentNode = currentNodeChildren[2]
                pathToLeaf.append(currentNode)
       
        def addKeyToNode(node,element):
            keys = node.getKeys()
            if len(keys) == 0 or element < keys[0]:
                node.addKey(element,0)
            elif len(keys) == 1 or element < keys[1]:
                node.addKey(element,1)
            else:
                node.addKey(element,2)
            return node.getKeys()
        
        pathToLeaf = getPathToLeaf(element)
        if pathToLeaf == False:
            return False
        
        for i in range(len(pathToLeaf)-1,-1,-1):
            currentNode = pathToLeaf[i]
            currentNodeKeys = addKeyToNode(currentNode,element)
            if len(currentNodeKeys) <= 2: #Adding to a 2-node
                return True
            currentNodeChildren = currentNode.getChildren()
            currentLeaf = currentNode.isLeaf()
            atRoot = currentNode == self._root 
            if atRoot: #Adding to a 3-node root node
                parentNode = Node(isLeaf=False)
                parentNode.addKey(currentNodeKeys[1])
                self._root = parentNode
                currentNodeIndex = 0
            else: #Adding to a 3-node
                parentNode = pathToLeaf[i-1]
                currentNodeIndex = parentNode.getChildIndex(currentNode) #To add the new nodes to the correct position 
                parentNode.removeChild(currentNode)

            node1 = Node(currentLeaf) #Create new nodes 
            node1.addKey(currentNodeKeys[0])
            node2 = Node(currentLeaf)
            node2.addKey(currentNodeKeys[2])
            if currentLeaf == False: #Connect the new nodes to the rest of the tree if not leaf
                node1.addChild(currentNodeChildren[0])
                node1.addChild(currentNodeChildren[1])
                node2.addChild(currentNodeChildren[2])
                node2.addChild(currentNodeChildren[3])
            
            parentNode.addChild(node1,currentNodeIndex) #Add nodes in correct spot, if left, positions 0,1, if right, positions 1,2
            parentNode.addChild(node2,currentNodeIndex+1)
            element = currentNodeKeys[1]
            if atRoot:
                return True
            

            
            
            

Use the cell below to implement the requested API by means of **AVL Tree** (if among your chosen data structure).

In [180]:
class AVLTree(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found  

Use the cell below to implement the requested API by means of **LLRB BST** (if among your chosen data structure).

In [ ]:
#Alfie 
from enum import Enum

class Color(Enum):
     BLACK = False
     RED = True


class Node:
    def __init__(self, key):
        self.key = key
        self.left = None
        self.right = None
        #False - black, True - red
        self.color = Color.BLACK  
    
    
class LLRBBST(AbstractSearchInterface):
    def __init__(self):
        self.root = None

    def __rotateLeft(self, n: Node):
        x = n.right
        n.right = x.left
        x.left = n
        x.color = n.color
        n.color = Color.RED

        return x
    
    def __rotateRight(self, n: Node):
        x = n.left
        n.left = x.right
        x.right = n
        x.color = n.color
        n.color = Color.RED

        return x
    
    def __flipColor(self, n: Node):
        if n.color == Color.BLACK: n.color = Color.RED 
        else: n.color = Color.BLACK
 
        if n.left:
            if n.left.color == Color.BLACK: n.left.color = Color.RED
            else: n.left.color = Color.BLACK
        if n.right: 
            if n.right.color == Color.BLACK: n.right.color = Color.RED
            else: n.right.color = Color.BLACK

    def __insertRecursion(self, element, n: Node):
        if n is None:
            newNode = Node(element)
            newNode.color = Color.RED
            return newNode
        
        if element == n.key: 
            return n
        
        if element < n.key: 
            n.left = self.__insertRecursion(element, n.left)
        elif element > n.key: 
            n.right = self.__insertRecursion(element, n.right)

        if (self.__isRed(n.right) and not self.__isRed(n.left)): n =  self.__rotateLeft(n)
        if (self.__isRed(n.left) and self.__isRed(n.left.left)): n =  self.__rotateRight(n)
        if (self.__isRed(n.left) and self.__isRed(n.right)): self.__flipColor(n)

        return n

    def __isRed(self, n: Node):
        return (n is not None and n.color == Color.RED)
         
    def insertElement(self, element):
        elementExists = self.searchElement(element)

        self.root = self.__insertRecursion(element, self.root)
        if self.root: self.root.color = Color.BLACK

        return not elementExists
    
    def printTree(self, n, level=0):
        if n is None:
            print("   " * level + "---")
            return

        print("   " * level + f"{n.key}: {n.color}")

        self.printTree(n.left, level + 1)
        self.printTree(n.right, level + 1)

    def searchElement(self, element):     
        found = False
        n = self.root
        
        while n is not None:
            if element == n.key: 
                found = True
                break 
            elif element < n.key: n = n.left
            else: n = n.right
        
        return found 


Use the cell below to implement the requested API by means of **B-Tree** (if among your chosen data structure).

In [182]:
class BTree(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found

Use the cell below to implement the requested API by means of **Scapegoat Tree** (if among your chosen data structure).

In [183]:
class ScapegoatTree(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found 

Use the cell below to implement the **synthetic data generator** needed by your experimental framework (be mindful of code readability and reusability).

In [184]:
# Alex
import string
import random

class TestDataGenerator():
    '''
    A class to represent a synthetic data generator.

    ...

    Attributes
    ----------
    
    [to be defined as part of the coursework]

    Methods
    -------
    
    [to be defined as part of the coursework]

    '''
    
    #ADD YOUR CODE HERE
    
    def __init__():
        pass
    

Use the cell below to implement the requested **experimental framework** (be mindful of code readability and reusability).

In [185]:
# Alex

import timeit
import matplotlib

class ExperimentalFramework():
    '''
    A class to represent an experimental framework.

    ...

    Attributes
    ----------
    
    [to be defined as part of the coursework]

    Methods
    -------
    
    [to be defined as part of the coursework]

    '''
            
    #ADD YOUR CODE HERE
    
    def __init__():
        pass
    

Use the cell below to illustrate the python code you used to **fully evaluate** your three chosen search data structures and algortihms. The code below should illustrate, for example, how you made used of the **TestDataGenerator** class to generate test data of various size and properties; how you instatiated the **ExperimentalFramework** class to  evaluate each data structure using such data, collect information about their execution time, plot results, etc. Any results you illustrate in the companion PDF report should have been generated using the code below.

In [186]:
# ADD YOUR TEST CODE HERE 



